# EDA 03 — Perhitungan Volatilitas Return Saham

Notebook ini digunakan untuk menghitung volatilitas return saham harian sebagai variabel target (Y) dalam penelitian tesis.

Analisis meliputi:
1. Import library dan konfigurasi path
2. Perhitungan rolling volatility (20 hari)
3. Statistik deskriptif volatilitas
4. Visualisasi time series volatilitas
5. Boxplot distribusi volatilitas antar emiten
6. Penyimpanan dataset hasil preprocessing

Output digunakan pada:
**BAB 4.3 Perhitungan Volatilitas Return (Variabel Y)**

In [ ]:
# ============================================================
# CELL 1 — IMPORT LIBRARY & KONFIGURASI PATH
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Konfigurasi visual
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['figure.dpi'] = 200
plt.rcParams['figure.dpi'] = 200

# Path project (karena notebook berada di 02_preprocessing/EDA)
ROOT_DIR = Path('../../')

PROCESSED_DIR = ROOT_DIR / '03_dataset' / 'processed'
OUTPUT_DIR = ROOT_DIR / '06_dokumen_tesis'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Warna emiten konsisten
URUT_EMITEN = ['ADRO', 'ADMR', 'ITMG', 'PTBA', 'MEDC', 'AKRA', 'PGAS']   # urutan sesuai Tabel 1.2 tesis

colors = {
    'ADRO': '#1f77b4',
    'ADMR': '#2ca02c',
    'ITMG': '#ff7f0e',
    'PTBA': '#d62728',
    'MEDC': '#9467bd',
    'AKRA': '#8c564b',
    'PGAS': '#e377c2'
}

print("Processed Folder:", PROCESSED_DIR.resolve())
print("Output Folder   :", OUTPUT_DIR.resolve())

## Perhitungan Rolling Volatility

Volatilitas *return* saham dihitung sebagai **standar deviasi *log return* dengan *rolling window* 20 hari perdagangan** (±1 bulan perdagangan), sesuai definisi variabel Y pada Bab 3.

$$\sigma_t = \sqrt{\frac{1}{n-1}\sum_{i=t-19}^{t}(R_i-\bar{R})^2}, \quad n = 20$$

Target pemodelan adalah volatilitas satu hari ke depan, $Y_t = \sigma_{t+1}$ (pergeseran dilakukan pada tahap integrasi dataset).

In [ ]:
# ============================================================
# CELL 2 — LOAD LOG RETURN & HITUNG VOLATILITAS
# ============================================================

INPUT_FILE = PROCESSED_DIR / 'log_return.csv'

# Load dataset
df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')

# Rapikan kolom
df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'])

# Pastikan urut kronologis
df = df.sort_values(
    ['Kode_Emiten', 'Date']
).reset_index(drop=True)

# ------------------------------------------------------------
# PARAMETER ROLLING WINDOW — HARUS SAMA DENGAN BAB 3 (20 hari)
# ------------------------------------------------------------
WINDOW = 20

# Hitung rolling volatility
df['Volatilitas'] = (
    df.groupby('Kode_Emiten')['Log_Return']
      .transform(
          lambda x: x.rolling(
              window=WINDOW,
              min_periods=WINDOW
          ).std()
      )
)

# Opsional: annualized volatility (jika nanti diperlukan)
# df['Volatilitas_Annualized'] = df['Volatilitas'] * np.sqrt(252)

print("="*60)
print(f"Rolling volatility berhasil dihitung (window={WINDOW} hari)")
print("="*60)

print("Shape dataset:", df.shape)
print("Jumlah missing akibat rolling:", df['Volatilitas'].isna().sum())

display(df[['Date','Kode_Emiten','Adj Close','Log_Return','Volatilitas']].iloc[18:25])


## Statistik Deskriptif Volatilitas
Output untuk tabel statistik deskriptif volatilitas di Bab 4 (Mean, Std, Min, Maks, dan volatilitas tahunan = σ × √252).

In [ ]:
# ============================================================
# CELL 3 — STATISTIK DESKRIPTIF VOLATILITAS
# ============================================================
summary = []
for kode in URUT_EMITEN:
    data = df.loc[df['Kode_Emiten'] == kode, 'Volatilitas'].dropna()
    summary.append({'Emiten': kode, 'N': len(data), 'Mean': data.mean(), 'Std Dev': data.std(),
                    'Min': data.min(), 'Maks': data.max(),
                    'Vol. Tahunan': data.mean() * np.sqrt(252),
                    'Tanggal Maks': df.loc[data.idxmax(), 'Date'].date()})
vol_table = pd.DataFrame(summary).round(6)
display(vol_table)
vol_table.to_csv(OUTPUT_DIR / f'Tabel_statistik_volatilitas_rolling{WINDOW}.csv', index=False, encoding='utf-8-sig')


## Grafik Time Series Volatilitas Rolling

Visualisasi ini digunakan untuk mengamati pola **volatility clustering**, yaitu kondisi ketika periode volatilitas tinggi cenderung diikuti oleh volatilitas tinggi.

Output digunakan sebagai **Gambar 4.7**

In [ ]:
# ============================================================
# CELL 4 — TIME SERIES VOLATILITAS
# ============================================================

emitens = URUT_EMITEN

# Canvas subplot
fig, axes = plt.subplots(4, 2, figsize=(15, 14))
axes = axes.flatten()

for i, kode in enumerate(emitens):
    ax = axes[i]

    # Filter data per emiten
    sub = (
        df[df['Kode_Emiten'] == kode]
        .dropna(subset=['Volatilitas'])
        .copy()
    )

    # Plot garis volatilitas
    ax.plot(
        sub['Date'],
        sub['Volatilitas'],
        color=colors[kode],
        linewidth=0.9
    )

    # Area fill supaya lebih estetik
    ax.fill_between(
        sub['Date'],
        sub['Volatilitas'],
        color=colors[kode],
        alpha=0.10
    )

    # Judul subplot
    ax.set_title(
        f"{kode}",
        fontsize=12,
        fontweight='bold'
    )

    ax.set_xlabel("Tanggal", fontsize=9)
    ax.set_ylabel("Volatilitas", fontsize=9)

    # Grid halus
    ax.grid(
        alpha=0.25,
        linestyle='--'
    )

    # Rotasi tanggal
    ax.tick_params(
        axis='x',
        rotation=45,
        labelsize=8
    )

    ax.tick_params(
        axis='y',
        labelsize=8
    )

    # Hilangkan border atas-kanan
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Hapus subplot kosong
for j in range(len(emitens), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"Gambar_time_series_volatilitas_rolling{WINDOW}.png", dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


## Boxplot Distribusi Volatilitas Antar Emiten

Boxplot digunakan untuk membandingkan:
- median volatilitas
- rentang interquartile
- outlier
- stabilitas relatif antar emiten

Output digunakan sebagai **Gambar 4.8**

In [ ]:
# ============================================================
# CELL 5 — BOXPLOT VOLATILITAS
# ============================================================

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df.dropna(subset=['Volatilitas']),
    x='Kode_Emiten',
    y='Volatilitas',
    order=URUT_EMITEN,
    palette=colors,
    width=0.6,
    linewidth=1.2
)


plt.xlabel("Kode Emiten")
plt.ylabel("Volatilitas")

plt.savefig(
    OUTPUT_DIR / f"Gambar_boxplot_volatilitas_rolling{WINDOW}.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## Simpan Dataset Volatilitas

Dataset volatilitas akan digunakan sebagai variabel target (Y) pada tahap feature engineering dan pemodelan.

In [ ]:
# ============================================================
# CELL 6 — SAVE DATASET
# ============================================================

df.to_csv(
    PROCESSED_DIR / f'volatilitas_rolling{WINDOW}.csv',
    index=False,
    encoding='utf-8-sig'
)

print("Dataset berhasil disimpan:")
print(PROCESSED_DIR / f'volatilitas_rolling{WINDOW}.csv')

## Ringkasan Output

Semua tabel, grafik, dan dataset volatilitas telah berhasil disimpan.

In [ ]:
print("Output dokumen:")
for f in [f'Tabel_statistik_volatilitas_rolling{WINDOW}.csv', f'Gambar_time_series_volatilitas_rolling{WINDOW}.png',
          f'Gambar_boxplot_volatilitas_rolling{WINDOW}.png']:
    print("-", OUTPUT_DIR / f)
print("Dataset:", PROCESSED_DIR / f'volatilitas_rolling{WINDOW}.csv')
